# 3 — Get Medoids for Dynamic Prompt

**Goal:** for each of the 8 clusters, identify the **medoid** — the population note whose embedding is closest to the cluster centroid — and freeze it as the single dynamic-prompt example for that cluster.

**Rationale:** the dynamic strategy conditions the model on the most representative note of the incoming note's cluster. The medoid is the natural choice: it is a *real* note (unlike the centroid itself, which is just a point in embedding space) and it maximises similarity to the cluster's typical content.

**Contamination control:** medoid candidates exclude the 32 sample notes (dev *and* prod). Examples therefore come strictly from the population, so no annotated evaluation note ever appears inside a prompt.


In [ ]:
from pathlib import Path
from clinical_notes_extraction.config import PROJECT_ROOT
import os

import numpy as np
import pandas as pd

In [ ]:
# Constants local to this notebook
DATA_PATH = Path(f"{PROJECT_ROOT}/scripts/3_information_extraction/3_2_medications_on_admission/data")

EMBEDDINGS_FILE = Path(f"{DATA_PATH}/embeddings.npy")        # population embeddings, row-aligned with POPULATION_FILE
POPULATION_FILE = Path(f"{DATA_PATH}/final_dataset.parquet")   # note_id, cluster, text (output of the clustering notebook)
SAMPLE_DIR = Path(f"{DATA_PATH}/sample.parquet")                     # to exclude sample notes from medoid candidates
OUTPUT_DIR = Path(f"{DATA_PATH}/annotations/dynamic_prompts")

# NOTE: the medoid search should run in the SAME representation used by KMeans.
# If clustering was done on the PCA-reduced, L2-renormalised matrix, point
# EMBEDDINGS_FILE at that matrix (or re-apply the saved PCA transform here).

## Load embeddings and population, exclude the annotated sample

Load the population notes and their Sentence-BERT embeddings, which were computed row-by-row
over this exact dataframe (hence the alignment assert). Two alignment fixes are applied:

- **Reset the population index** — the parquet could have kept stale index labels from an earlier
  filtering step, and the medoid selection uses those labels to positionally slice the
  embeddings array.
- **Filter dataframe and embeddings with the same boolean mask** — the 32 annotated sample
  notes (dev + prod) are removed from both, keeping them row-aligned by construction. **Excluding the sample prevents contamination: a note used as a dynamic prompt example must
never be one of the notes the LLM is evaluated on.**

In [ ]:
embeddings = np.load(EMBEDDINGS_FILE)
population = pd.read_parquet(POPULATION_FILE)

assert len(embeddings) == len(population), (
    "Embeddings and population dataframe must be row-aligned"
)

# The population parquet kept stale index labels from an earlier filtering step.
# Reset it so index labels match row positions in the embeddings array.
population = population.reset_index(drop=True)

# All 32 sample notes (dev + prod) are excluded as medoid candidates,
# so no note ever appears both as a dynamic prompt example and as an evaluation note.
sample_ids = pd.read_parquet(SAMPLE_DIR)["note_id"]

# Filter dataframe and embeddings with the same boolean mask so both stay row-aligned.
mask = ~population["note_id"].isin(sample_ids)
cleaned_population = population[mask].reset_index(drop=True)
cleaned_embeddings = embeddings[mask.to_numpy()]

assert len(cleaned_population) == len(cleaned_embeddings)

print(
    f"{len(population)} population notes | "
    f"{len(sample_ids)} sample notes excluded | "
    f"{len(cleaned_population)} cleaned population notes that will be used to get the medoids"
)

## Find the medoid of each cluster

For each cluster, the **centroid** is the mean of its members' embeddings — an abstract point
that does not correspond to any real note. The **medoid** is the actual note closest to that
point, which is what we need as a concrete example for the dynamic prompt.

Steps:

1. **L2-normalize** all embeddings so that the dot product equals cosine similarity
   (direction carries the semantics; vector length is irrelevant).
2. **Compute the centroid** of each cluster and re-normalize it — the mean of unit vectors
   is *not* a unit vector, so without this step the similarities would no longer be true
   cosines and would not be comparable across clusters.
3. **Rank members** by cosine similarity to the centroid and pick the most central note
   that is not part of the annotated sample (dev + prod), avoiding contamination
   between few-shot examples and evaluation data.

Note: this is the standard "closest to centroid" approximation of the medoid — linear cost
instead of the quadratic pairwise-distance computation, with virtually identical results.

In [ ]:
def l2_normalize(matrix: np.ndarray) -> np.ndarray:
    # Divide each vector by its length so all vectors have norm 1.
    # With unit vectors, the dot product equals cosine similarity.
    # The clip avoids division by zero for degenerate all-zero vectors.
    norms = np.linalg.norm(matrix, axis=-1, keepdims=True)
    return matrix / np.clip(norms, 1e-12, None)


# cleaned_population and cleaned_embeddings were filtered with the same mask
# and re-indexed, so index labels are true row positions in the array.
normed = l2_normalize(cleaned_embeddings)
medoids = {}

for cluster_id, group in cleaned_population.groupby("cluster"):
    # Row positions of this cluster's members in the embeddings array.
    member_idx = group.index.to_numpy()

    # Centroid = mean of the members' embeddings, re-normalized because the
    # mean of unit vectors is not a unit vector.
    centroid = l2_normalize(normed[member_idx].mean(axis=0))

    # Cosine similarity of every member to the centroid (one value per note).
    """ 
    Cosine similarity of every member to the centroid.
    The @ operator is matrix multiplication: 
        each row of normed[member_idx] (shape n_members × 384) is multiplied by the centroid 
        vector (shape 384), yielding one dot product per note.
    Since both are L2-normalized, this dot product equals cosine similarity.
    """
    similarities = normed[member_idx] @ centroid

    # Sample notes are already excluded, so the most central member is the medoid.
    best = np.argmax(similarities)
    row = cleaned_population.loc[member_idx[best]]
    medoids[int(cluster_id)] = {
        "note_id": row["note_id"],
        "similarity_to_centroid": float(similarities[best]),
        "text": row["text"],
    }

# Compact overview: one row per cluster with its medoid and how central it is.
summary = pd.DataFrame(
    {c: {"note_id": m["note_id"], "similarity": round(m["similarity_to_centroid"], 3)}
     for c, m in medoids.items()}
).T
summary.index.name = "cluster"
summary

In [ ]:
# Filter cleaned_population to keep only the rows whose note_id is in the medoid list,
# then select the specific columns needed for the dynamic prompt.
cols = [
    'note_id', 'subject_id', 'text', 'meds_on_admission_cleaned',
    'meds_on_admission_cleaned_length', 'cluster',
    'meds_on_admission_cleaned_length_classification',
    'meds_on_admission_cleaned_length_binary'
]

dynamic_prompt_medoid = cleaned_population[
    cleaned_population['note_id'].isin(summary['note_id'].values)
    ][cols]

dynamic_prompt_medoid.sort_values(by='cluster', ascending=True)

## Annotate these 8 medoids notes manually and then add these annotations as a new column of the dataset

- Each medoid gets an empty `medications` list. **Annotate these 8 notes manually** (verbatim `span` per medication, same schema as the ground truth) before running the dynamic strategy — the extraction script will fail loudly on an empty example, which is intended.
- Add annotation_json medication as a new column containing the annotation of the medication on admission for these medoids notes

In [ ]:
dynamic_prompt_annotations = pd.read_json(f'{OUTPUT_DIR}/dynamic_prompt_medoids_annotations.json')

dynamic_prompt_annotations

In [ ]:
dynamic_prompt_df = dynamic_prompt_medoid.copy()

annotation_map = dynamic_prompt_annotations.set_index("note_id", drop=False).to_dict(orient="index")

dynamic_prompt_df["annotation_json"] = dynamic_prompt_df["note_id"].map(annotation_map)

dynamic_prompt_df

## Add embedding as a new column
- During dynamic prompts we will need to have the embedding value of the medoids to get the best example to be passed in prompt.

In [ ]:
dynamic_prompt_note_embeddings = pd.Series(
    cleaned_population.index, index=cleaned_population["note_id"]
).to_dict()

dynamic_prompt_df["embedding"] = dynamic_prompt_df["note_id"].map(
    lambda nid: cleaned_embeddings[dynamic_prompt_note_embeddings[nid]] if nid in dynamic_prompt_note_embeddings else None
)

dynamic_prompt_df



In [ ]:
final_dynamic_prompt_df = dynamic_prompt_df[["note_id", "text", "cluster", "annotation_json", "embedding"]]

final_dynamic_prompt_df

## Save dataset containing medoids annotations

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

final_dynamic_prompt_df.to_parquet(f"{OUTPUT_DIR}/dynamic_prompt_medoids_annotations.parquet")